In [2]:
"""
Growth curve parameter estimation v4
======================================
Pipeline per replicate
-----------------------
1. Logistic-first  : 3 start points → TRF → 6-point sanity check
2. Fallback cascade (if logistic fails sanity):
       Gompertz → Richards → Exponential → Biphasic
       all scored by AICc; lowest wins
3. Jacobian CI     : 95% CI from TRF Jacobian — no extra packages needed
 
Output TSV columns
------------------
ID, model, L, r, t0, L0, delta_H, lag,
L_lo, L_hi, r_lo, r_hi, t0_lo, t0_hi, L0_lo, L0_hi,
AICc, converged, raw_params  (JSON — exact params for curve reconstruction)
"""

'\nGrowth curve parameter estimation v4\n======================================\nPipeline per replicate\n-----------------------\n1. Logistic-first  : 3 start points → TRF → 6-point sanity check\n2. Fallback cascade (if logistic fails sanity):\n       Gompertz → Richards → Exponential → Biphasic\n       all scored by AICc; lowest wins\n3. Jacobian CI     : 95% CI from TRF Jacobian — no extra packages needed\n \nOutput TSV columns\n------------------\nID, model, L, r, t0, L0, delta_H, lag,\nL_lo, L_hi, r_lo, r_hi, t0_lo, t0_hi, L0_lo, L0_hi,\nAICc, converged, raw_params  (JSON — exact params for curve reconstruction)\n'

In [4]:
import json, warnings, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.signal import savgol_filter, find_peaks
 
warnings.filterwarnings("ignore")
np.random.seed(100)

### Inputs

In [5]:
growth     = Path("../../data/For_statistical_analysis_strains_2.xlsx")
out_params = Path("../../out/Growth_features_estimates_2.tsv")
out_failed = Path("../../out/Growth_features_estimates_failed_2.tsv")
out_params.parent.mkdir(parents=True, exist_ok=True)
 
COLS = ["ID","model","L","r","t0","L0","delta_H","lag",
        "L_lo","L_hi","r_lo","r_hi","t0_lo","t0_hi","L0_lo","L0_hi",
        "AICc","converged","raw_params"]
out_params.write_text("\t".join(COLS) + "\n")
out_failed.write_text("ID\n")

3

In [6]:
gd = pd.read_excel(growth, index_col=0)
gd.index.name = "ID_FINAL"
gd_l = gd.reset_index().melt(id_vars="ID_FINAL", var_name="Hours", value_name="OD")
gd_l["Hours"] = gd_l["Hours"].astype(float)
gd_l[["Sample","Replicate"]] = gd_l["ID_FINAL"].str.rsplit(".", n=1, expand=True)
gd_l["OD"] = gd_l["OD"].clip(lower=0)
for col in gd_l.select_dtypes(include=["object","string"]).columns:
    gd_l[col] = gd_l[col].astype(str).str.strip()

### Model functions

In [7]:
def lgf(L, r, t0, L0, t):
    """4-parameter logistic. Param order: L, r, t0, L0 (original convention)."""
    return L0 + L / (1.0 + np.exp(-np.clip(r*(t - t0), -300, 300)))
 
def gomp(t, L0, A, r, t0):
    """Gompertz — asymmetric sigmoid, left-shifted inflection."""
    return L0 + A * np.exp(-np.exp(-np.clip(r*(t - t0), -300, 300)))
 
def rich(t, L0, A, r, t0, v):
    """Richards — generalised logistic, variable inflection position."""
    v = max(float(v), 1e-4)
    return L0 + A / (1.0 + v * np.exp(-np.clip(r*(t-t0), -300, 300)))**(1.0/v)
 
def exps(t, L0, A, r):
    """Saturating exponential — for monotone / no-plateau curves."""
    return L0 + A * (1.0 - np.exp(-np.clip(r*t, 0, 300)))
 
def biphase(t, L0, A1, r1, t1, A2, r2, t2):
    """Two logistic phases sharing baseline L0 — diauxic / biphasic growth."""
    return (L0
            + A1 / (1.0 + np.exp(-np.clip(r1*(t - t1), -300, 300)))
            + A2 / (1.0 + np.exp(-np.clip(r2*(t - t2), -300, 300))))

In [8]:
def aicc(ssr, n, k):
    if n <= k+1 or ssr <= 0:
        return np.inf
    ll = -n/2.0 * (np.log(2*np.pi*ssr/n) + 1)
    return -2*ll + 2*k + 2*k*(k+1)/(n-k-1)

In [9]:
def compute_mask(h, od):
    """Keep rising phase + 5% plateau tail."""
    mi   = np.nanargmax(od); mt = h[mi]
    mm   = (od <= od.max()) & (h <= mt) & (h >= 0)
    tail = (np.abs(od - od.max()) / max(od.max(), 1e-9) <= 0.05) & (h > mt)
    keep = mm | tail
    return keep if keep.sum() >= 4 else np.ones(len(od), bool)
 
def _bl(od):  return max(float(np.percentile(od, 5)), 0.0)
def _amp(od): return max(float(np.nanmax(od)) - _bl(od), 0.01)
 
def _biphasic_t_starts(h, od):
    """Data-driven starting estimates for the two inflection times."""
    try:
        od_s = savgol_filter(od, window_length=min(7, len(od)|1), polyorder=2)
        d1   = np.gradient(od_s, h)
        pks, _ = find_peaks(d1, height=0.01*d1.max(), distance=3)
        if len(pks) >= 2:
            return float(h[pks[0]]), float(h[pks[1]])
        elif len(pks) == 1:
            t1 = float(h[pks[0]])
            return t1, min(t1 + 6.0, float(h.max()) - 1.0)
    except Exception:
        pass
    return float(h.max()) * 0.3, float(h.max()) * 0.7

In [10]:
def _trf(resid_fn, x0_list, lo, hi):
    """Try each start with TRF; return best (par, jac, residuals)."""
    best_ssr, best_par, best_jac, best_res = np.inf, None, None, None
    for x0 in x0_list:
        try:
            r = least_squares(resid_fn, np.clip(x0, lo, hi),
                              bounds=(lo, hi), method="trf",
                              max_nfev=30000, ftol=1e-12, xtol=1e-12,
                              jac="3-point")
            s = float(np.sum(r.fun**2))
            if s < best_ssr:
                best_ssr, best_par = s, r.x.copy()
                best_jac, best_res = r.jac, r.fun
        except Exception:
            pass
    if best_par is None:
        best_par = np.clip(x0_list[0], lo, hi)
    return best_par, best_jac, best_res
 
def _jac_ci(J, res, par, names, t95=2.0):
    """95% CI from Jacobian covariance C = σ²(JᵀJ)⁻¹."""
    if J is None or res is None:
        return {k: (np.nan, np.nan) for k in names}
    n = len(res); k = len(par)
    s2 = float(np.sum(res**2)) / max(n - k, 1)
    try:
        cov = np.linalg.inv(J.T @ J) * s2
        se  = np.sqrt(np.maximum(np.diag(cov), 0))
        return {nm: (float(par[i] - t95*se[i]), float(par[i] + t95*se[i]))
                for i, nm in enumerate(names)}
    except Exception:
        return {k: (np.nan, np.nan) for k in names}

In [11]:
def fit_logistic(h, od):
    A0=_amp(od); L0v=_bl(od); tm=h[np.nanargmax(od)]
    lo=[0.001, 0.05, 0.0,       0.0]
    hi=[A0*3,  15.0, h.max()+5, 0.5]
    x0s=[[A0,     1.0, tm,      L0v],
         [A0,     0.5, tm*0.85, L0v],
         [A0*0.8, 2.0, tm,      L0v]]
    par, J, res = _trf(lambda p: od - lgf(p[0],p[1],p[2],p[3],h), x0s, lo, hi)
    ssr = float(np.sum((od - lgf(*par, h))**2))
    return dict(model="logistic4", par=par, keys=["L","r","t0","L0"], n=4,
                ssr=ssr, aicc=aicc(ssr,len(od),5),
                ci=_jac_ci(J, res, par, ["L","r","t0","L0"]))
 
def fit_gompertz(h, od):
    A0=_amp(od); L0v=_bl(od); tm=h[np.nanargmax(od)]
    lo=[0.0, 0.001, 0.01, 0.0       ]
    hi=[0.5, A0*3,  15.0, h.max()+5 ]
    x0s=[[L0v, A0,     0.4, tm    ],
         [L0v, A0,     0.8, tm*0.8],
         [L0v, A0*0.8, 1.5, tm    ]]
    par, J, res = _trf(lambda p: od - gomp(h,*p), x0s, lo, hi)
    ssr = float(np.sum((od - gomp(h,*par))**2))
    return dict(model="gompertz4", par=par, keys=["L0","A","r","t0"], n=4,
                ssr=ssr, aicc=aicc(ssr,len(od),5),
                ci=_jac_ci(J, res, par, ["L0","A","r","t0"]))
 
def fit_richards(h, od):
    A0=_amp(od); L0v=_bl(od); tm=h[np.nanargmax(od)]
    lo=[0.0, 0.001, 0.01, 0.0,       0.01]
    hi=[0.5, A0*3,  15.0, h.max()+5, 10.0]
    x0s=[[L0v, A0,  1.0, tm, 1.0],
         [L0v, A0,  0.5, tm, 0.5],
         [L0v, A0,  2.0, tm, 2.0]]
    par, J, res = _trf(lambda p: od - rich(h,*p), x0s, lo, hi)
    ssr = float(np.sum((od - rich(h,*par))**2))
    return dict(model="richards", par=par, keys=["L0","A","r","t0","v"], n=5,
                ssr=ssr, aicc=aicc(ssr,len(od),6),
                ci=_jac_ci(J, res, par, ["L0","A","r","t0","v"]))
 
def fit_expsat(h, od):
    A0=_amp(od); L0v=_bl(od)
    lo=[0.0, 0.001, 0.001]
    hi=[0.5, A0*3,  5.0  ]
    x0s=[[L0v, A0,     0.15],
         [L0v, A0,     0.30],
         [L0v, A0*0.8, 0.08]]
    par, J, res = _trf(lambda p: od - exps(h,*p), x0s, lo, hi)
    ssr = float(np.sum((od - exps(h,*par))**2))
    return dict(model="exponential", par=par, keys=["L0","A","r"], n=3,
                ssr=ssr, aicc=aicc(ssr,len(od),4),
                ci=_jac_ci(J, res, par, ["L0","A","r"]))
 
def fit_biphasic(h, od):
    """
    Two sequential logistic terms sharing L0.
    Start points are data-driven from peaks in the smoothed first derivative.
    Only called when the logistic sanity check fails — AICc then decides
    whether biphasic genuinely fits better than the other fallbacks.
    """
    A0=_amp(od); L0v=_bl(od); tm=h[np.nanargmax(od)]
    t1g, t2g = _biphasic_t_starts(h, od)
 
    # enforce t1 < t2 and both inside data range
    t1g = float(np.clip(t1g, h.min(), tm - 0.5))
    t2g = float(np.clip(t2g, t1g + 1.0, h.max() + 2.0))
 
    lo=[0.0,  0.001, 0.05, h.min(), 0.001, 0.05, t1g+0.5  ]
    hi=[0.5,  A0*2,  15.0, tm,      A0*2,  15.0, h.max()+3 ]
    x0s=[
        [L0v, A0*0.5, 1.0, t1g,   A0*0.5, 0.8, t2g],
        [L0v, A0*0.4, 1.5, t1g,   A0*0.6, 0.5, t2g],
        [L0v, A0*0.6, 0.8, t1g,   A0*0.4, 1.2, t2g],
    ]
    par, J, res = _trf(lambda p: od - biphase(h,*p), x0s, lo, hi)
    ssr = float(np.sum((od - biphase(h,*par))**2))
    return dict(model="biphasic",
                par=par, keys=["L0","A1","r1","t1","A2","r2","t2"], n=7,
                ssr=ssr, aicc=aicc(ssr,len(od),8),
                ci=_jac_ci(J, res, par, ["L0","A1","r1","t1","A2","r2","t2"]))

In [8]:
def estimate_lag(hours, model, par):
    dense_t = np.linspace(hours.min(), hours.max(), 10000)
    if model == "logistic4":
        L, r, t0, L0 = par
        ys  = lgf(L, r, t0, L0, dense_t)
        thr = max(L0 + 0.05 * L, 0.01)
    elif model == "gompertz4":
        ys  = gompertz4(dense_t, *par)
        thr = par[0] + 0.05 * par[1]
    elif model == "richards":
        ys  = richards(dense_t, *par)
        thr = par[0] + 0.05 * par[1]
    elif model == "exponential":
        ys  = exponential_sat(dense_t, *par)
        thr = par[0] + 0.05 * par[1]
    elif model == "biphasic":
        ys  = biphasic(dense_t, *par)
        thr = par[0] + 0.05 * (par[1] + par[4])
    else:
        return np.nan
    idxs = np.where(ys >= thr)[0]
    return float(dense_t[idxs[0]]) if len(idxs) > 0 else np.nan

### Sanity check

In [12]:
def logistic_ok(par, h, od):
    L, r, t0, L0 = par
    ssr    = float(np.sum((od - lgf(L,r,t0,L0,h))**2))
    ss_tot = float(np.var(od)*len(od)) if np.var(od) > 0 else 1e-9
    return all([
        r   > 0.05,                        # positive growth rate
        lgf(L,r,t0,L0,40) < 3.0,          # no runaway extrapolation
        ssr < 0.5 * ss_tot,               # explains > 50% variance
        h.min()-5 <= t0 <= h.max()+5,     # inflection near data window
        L   > 0.005,                       # meaningful amplitude
        L0  >= 0,                          # non-negative baseline
    ])

In [13]:
def looks_biphasic(h, od):
    """
    Checks for ≥2 distinct peaks in the smoothed first derivative.
    Only triggers the biphasic model attempt — AICc makes final call.
    """
    if len(od) < 10:
        return False
    try:
        od_s = savgol_filter(od, window_length=min(7, len(od)|1), polyorder=2)
        d1   = np.gradient(od_s, h)
        pks, _ = find_peaks(d1, height=0.05*d1.max(), distance=3)
        return len(pks) >= 2
    except Exception:
        return False

In [12]:
def lmfit_ci(model_name, par_arr, par_keys, hours, od, sigma=0.95):
    """
    Refit with lmfit and compute profile-likelihood confidence intervals.
    Returns dict {param: (lo, hi)} or NaNs on failure.
    """
    nan_ci = {k: (np.nan, np.nan) for k in par_keys}
    try:
        params = lmfit.Parameters()

        if model_name == "logistic4":
            L, r, t0, L0 = par_arr
            params.add("L",  value=L,  min=0.01, max=5.0)
            params.add("r",  value=r,  min=0.01, max=15.0)
            params.add("t0", value=t0, min=-10,  max=100.0)
            params.add("L0", value=L0, min=0.0,  max=2.0)
            def resid(p): return od - lgf(p["L"], p["r"], p["t0"], p["L0"], hours)

        elif model_name == "gompertz4":
            L0, A, r, t0 = par_arr
            params.add("L0", value=L0, min=0.0, max=2.0)
            params.add("A",  value=A,  min=0.01,max=5.0)
            params.add("r",  value=r,  min=0.01,max=15.0)
            params.add("t0", value=t0, min=-10, max=100.0)
            def resid(p): return od - gompertz4(hours, p["L0"],p["A"],p["r"],p["t0"])

        elif model_name == "richards":
            L0, A, r, t0, v = par_arr
            params.add("L0", value=L0, min=0.0, max=2.0)
            params.add("A",  value=A,  min=0.01,max=5.0)
            params.add("r",  value=r,  min=0.01,max=15.0)
            params.add("t0", value=t0, min=-10, max=100.0)
            params.add("v",  value=v,  min=0.01,max=10.0)
            def resid(p): return od - richards(hours,p["L0"],p["A"],p["r"],p["t0"],p["v"])

        elif model_name == "exponential":
            L0, A, r = par_arr
            params.add("L0", value=L0, min=0.0, max=2.0)
            params.add("A",  value=A,  min=0.01,max=5.0)
            params.add("r",  value=r,  min=1e-4,max=10.0)
            def resid(p): return od - exponential_sat(hours, p["L0"],p["A"],p["r"])

        elif model_name == "biphasic":
            L0,A1,r1,t1,A2,r2,t2 = par_arr
            params.add("L0", value=L0, min=0.0, max=2.0)
            params.add("A1", value=A1, min=0.01,max=5.0)
            params.add("r1", value=r1, min=0.01,max=15.0)
            params.add("t1", value=t1, min=0,   max=50.0)
            params.add("A2", value=A2, min=0.01,max=5.0)
            params.add("r2", value=r2, min=0.01,max=15.0)
            params.add("t2", value=t2, min=0,   max=100.0)
            def resid(p): return od - biphasic(hours,p["L0"],p["A1"],p["r1"],p["t1"],
                                                           p["A2"],p["r2"],p["t2"])
        else:
            return nan_ci

        result = lmfit.minimize(resid, params, method="leastsq")

        try:
            ci = lmfit.conf_interval(result, result, sigmas=[sigma],
                                     verbose=False, maxiter=200)
            out = {}
            for k in par_keys:
                if k in ci:
                    vals = [v for (s, v) in ci[k]]
                    out[k] = (float(vals[0]), float(vals[-1]))
                else:
                    out[k] = (np.nan, np.nan)
            return out
        except Exception:
            # fall back to covariance-based SE ±2σ
            out = {}
            for k in par_keys:
                if k in result.params and result.params[k].stderr is not None:
                    se = result.params[k].stderr
                    v  = result.params[k].value
                    out[k] = (v - 2*se, v + 2*se)
                else:
                    out[k] = (np.nan, np.nan)
            return out

    except Exception:
        return nan_ci

### Master fit

In [14]:
def fit_one(h, od):
    """
    Fitting strategy:
    - Logistic always runs first (fast, original approach).
    - If logistic passes sanity AND the curve does NOT look biphasic → accept it.
    - If logistic passes sanity BUT the curve looks biphasic → run AICc competition:
        logistic vs biphasic; winner is returned.
    - If logistic fails sanity → full fallback cascade:
        Gompertz, Richards, Exponential, and Biphasic (if shape suggests it),
        all scored by AICc.
    """
    log_res = None
    try:
        log_res = fit_logistic(h, od)
        log_sane = logistic_ok(log_res["par"], h, od)
    except Exception:
        log_sane = False
 
    is_biphasic_shape = looks_biphasic(h, od)
 
    # ── Fast path: logistic good AND no biphasic hint ─────────────────────────
    if log_sane and not is_biphasic_shape:
        return log_res
 
    # ── Biphasic competition: logistic sane but shape hints at two phases ─────
    if log_sane and is_biphasic_shape:
        try:
            bi_res = fit_biphasic(h, od)
            # AICc decides: biphasic needs Δ < 0 (lower = better) to win
            if bi_res and bi_res["aicc"] < log_res["aicc"]:
                return bi_res
            return log_res          # logistic wins on parsimony
        except Exception:
            return log_res          # biphasic failed; keep logistic
 
    # ── Full fallback cascade (logistic failed sanity) ────────────────────────
    candidates = []
    for fn in [fit_gompertz, fit_richards, fit_expsat]:
        try:
            c = fn(h, od)
            if c:
                candidates.append(c)
        except Exception:
            pass
 
    if is_biphasic_shape:
        try:
            c = fit_biphasic(h, od)
            if c:
                candidates.append(c)
        except Exception:
            pass
 
    # keep failed logistic too — it may still win on AICc
    if log_res is not None:
        candidates.append(log_res)
 
    if not candidates:
        return None
    return min(candidates, key=lambda c: (c["aicc"], c["ssr"]))

### Lag estimator

In [15]:
def get_lag(h, model, par):
    """First time fitted curve exceeds L0 + 5% × amplitude."""
    dt = np.linspace(h.min(), h.max(), 5000)
    if   model == "logistic4":
        ys=lgf(par[0],par[1],par[2],par[3],dt); thr=max(par[3]+0.05*par[0],0.01)
    elif model == "gompertz4":
        ys=gomp(dt,*par); thr=par[0]+0.05*par[1]
    elif model == "richards":
        ys=rich(dt,*par); thr=par[0]+0.05*par[1]
    elif model == "exponential":
        ys=exps(dt,*par); thr=par[0]+0.05*par[1]
    elif model == "biphasic":
        ys=biphase(dt,*par); thr=par[0]+0.05*(par[1]+par[4])
    else:
        return np.nan
    idx = np.where(ys >= thr)[0]
    return float(dt[idx[0]]) if len(idx) > 0 else np.nan

### Output

In [16]:
def unify(model, par, ci):
    """Map any model's raw params to the shared output columns."""
    def _g(k): return ci.get(k, (np.nan, np.nan))
 
    if model == "logistic4":
        L,r,t0,L0 = par
        Lci,Rci,T0ci,L0ci = _g("L"),_g("r"),_g("t0"),_g("L0")
 
    elif model in ("gompertz4","richards"):
        L0,L,r,t0 = par[0],par[1],par[2],par[3]
        Lci,Rci,T0ci,L0ci = _g("A"),_g("r"),_g("t0"),_g("L0")
 
    elif model == "exponential":
        L0,L,r,t0 = par[0],par[1],par[2],0.0
        Lci,Rci,T0ci,L0ci = _g("A"),_g("r"),(np.nan,np.nan),_g("L0")
 
    elif model == "biphasic":
        L0 = par[0]
        L  = par[1] + par[4]               # total amplitude
        # report dominant phase (larger amplitude)
        if par[1] >= par[4]:
            r,t0 = par[2],par[3]
            Rci  = _g("r1"); T0ci = _g("t1")
        else:
            r,t0 = par[5],par[6]
            Rci  = _g("r2"); T0ci = _g("t2")
        Lci  = (float(par[1])+float(_g("A1")[0]), float(par[1])+float(_g("A2")[0]))
        # approximate combined CI: sum of A1+A2 intervals
        A1lo,A1hi = _g("A1"); A2lo,A2hi = _g("A2")
        Lci  = (float(A1lo)+float(A2lo), float(A1hi)+float(A2hi)) \
               if all(np.isfinite([A1lo,A1hi,A2lo,A2hi])) else (np.nan,np.nan)
        L0ci = _g("L0")
    else:
        L,r,t0,L0 = par[1],par[2],par[3],par[0]
        Lci=Rci=T0ci=L0ci=(np.nan,np.nan)
 
    return L,r,t0,L0, Lci,Rci,T0ci,L0ci

### Main loop

In [17]:
samples = [s for s in gd_l["Sample"].unique() if s != "Blanc"]
total   = sum(len(gd_l[gd_l["Sample"]==s]["ID_FINAL"].unique()) for s in samples)
done    = 0
model_counts = Counter()
 
print(f"Fitting {total} replicates across {len(samples)} samples ...")
 
for sample_id in samples:
    sel_all = gd_l[gd_l["Sample"] == sample_id].sort_values("Hours")
 
    for rep in sel_all["ID_FINAL"].unique():
        sel = sel_all[sel_all["ID_FINAL"] == rep].sort_values("Hours")
        h   = sel["Hours"].values.astype(float)
        od  = sel["OD"].values.astype(float)
        done += 1
        if done % 100 == 0:
            print(f"  {done}/{total}")
 
        if od.max() < 0.01:
            with out_failed.open("a") as fh: fh.write(f"{rep}\n")
            continue
 
        keep = compute_mask(h, od)
        fh_  = h[keep]  if keep.sum() >= 4 else h
        fo_  = od[keep] if keep.sum() >= 4 else od
 
        res  = fit_one(fh_, fo_)
        if res is None:
            with out_failed.open("a") as fh: fh.write(f"{rep}\n")
            continue
 
        model = res["model"]; par = res["par"]; ci = res["ci"]
        raw   = dict(zip(res["keys"], [float(x) for x in par]))
        lag   = get_lag(h, model, par)
        L,r,t0,L0, Lci,Rci,T0ci,L0ci = unify(model, par, ci)
        model_counts[model] += 1
 
        row = dict(
            ID=rep, model=model, L=L, r=r, t0=t0, L0=L0,
            delta_H=float(L + L0), lag=lag,
            L_lo=Lci[0],  L_hi=Lci[1],
            r_lo=Rci[0],  r_hi=Rci[1],
            t0_lo=T0ci[0],t0_hi=T0ci[1],
            L0_lo=L0ci[0],L0_hi=L0ci[1],
            AICc=res["aicc"], converged=1,
            raw_params=json.dumps(raw),
        )
        with out_params.open("a") as fh:
            fh.write("\t".join(str(row[k]) for k in COLS) + "\n")
 
print(f"\nDone.")
print(f"  Params  → {out_params}")
print(f"  Failed  → {out_failed}")
print(f"\nModel usage:")
total_fit = sum(model_counts.values())
for m, c in model_counts.most_common():
    print(f"  {m}: {c}  ({100*c/total_fit:.1f}%)")

Fitting 718 replicates across 359 samples ...
  100/718
  200/718
  300/718
  400/718
  500/718
  600/718
  700/718

Done.
  Params  → ../../out/Growth_features_estimates_2.tsv
  Failed  → ../../out/Growth_features_estimates_failed_2.tsv

Model usage:
  logistic4: 573  (84.3%)
  biphasic: 105  (15.4%)
  exponential: 2  (0.3%)
